# Etapa 1: Ingesta Raw y Validación Inicial

Este cuaderno realiza la validación inicial de presencia y consistencia de los archivos CSV originales de entrada colocados en la carpeta `data/raw/`.

In [1]:
import sys
import os
from pathlib import Path
import pandas as pd

# Resolver la ruta raíz del proyecto de forma dinámica y portable
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyarrow", "pandas", "scikit-learn", "joblib", "numpy"])
    print("Dependencias instaladas en Colab.")
    
    # Intentar montar Google Drive automáticamente si no está montado
    if not Path('/content/drive').exists():
        try:
            from google.colab import drive
            drive.mount('/content/drive')
        except Exception as e:
            print("No se pudo montar Drive automáticamente. Por favor, móntelo en el panel izquierdo de Colab.")

current_dir = Path(os.getcwd()).resolve()
if IN_COLAB:
    # Rutas de búsqueda comunes en Google Drive y Colab
    possible_paths = [
        Path('/content/drive/MyDrive/Colab Notebooks/Proyecto'),
        Path('/content/drive/MyDrive/Proyecto'),
        Path('/content/Proyecto'),
        Path('/content')
    ]
    for p in possible_paths:
        if (p / "notebooks").exists():
            current_dir = p / "notebooks"
            break

if current_dir.name == "notebooks":
    PROJECT_ROOT = current_dir.parent
else:
    PROJECT_ROOT = current_dir

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))
raw_dir = PROJECT_ROOT / "data" / "raw"
print("Ruta Raw de entrada:", raw_dir)
print("¿Se ejecuta en Google Colab?:", IN_COLAB)

Ruta Raw de entrada: /Volumes/HD/0.2.Sistemas_de_Informacion_UG/11vo-semestre/ANÁLISIS DE DATOS MASIVO/Proyecto/data/raw
¿Se ejecuta en Google Colab?: False


## 1. Validación de Existencia de Archivos

In [2]:
required_files = [
    "clientes.csv",
    "provincias.csv",
    "segmentos.csv",
    "productos.csv",
    "cliente_estado_mensual.csv",
    "cliente_producto_mensual.csv",
    "cliente_producto_alta.csv"
]

missing_files = []
for f in required_files:
    f_path = raw_dir / f
    if f_path.exists():
        print(f"✔ Encontrado: {f} (Tamaño: {f_path.stat().st_size / 1024:.2f} KB)")
    else:
        print(f"❌ FALTANTE: {f}")
        missing_files.append(f)

if len(missing_files) == 0:
    print("\n¡Todos los archivos requeridos están presentes en la capa Raw!")
else:
    print(f"\nERROR: Faltan {len(missing_files)} archivos en la capa Raw.")

✔ Encontrado: clientes.csv (Tamaño: 74.32 KB)
✔ Encontrado: provincias.csv (Tamaño: 0.66 KB)
✔ Encontrado: segmentos.csv (Tamaño: 0.09 KB)
✔ Encontrado: productos.csv (Tamaño: 0.79 KB)
✔ Encontrado: cliente_estado_mensual.csv (Tamaño: 120.01 KB)
✔ Encontrado: cliente_producto_mensual.csv (Tamaño: 475.65 KB)
✔ Encontrado: cliente_producto_alta.csv (Tamaño: 0.07 KB)

¡Todos los archivos requeridos están presentes en la capa Raw!


## 2. Inspección Inicial de Dimensiones y Muestreo
Cargamos muestras de cada tabla para analizar el número de registros y verificar los nombres de las columnas.

In [3]:
for f in required_files:
    f_path = raw_dir / f
    if f_path.exists():
        df = pd.read_csv(f_path, nrows=5)
        # Leer total de registros de forma rápida
        total_rows = sum(1 for _ in open(f_path)) - 1
        print("=" * 60)
        print(f"Tabla: {f} | Registros: {total_rows} | Columnas: {list(df.columns)}")
        display(df.head(2))

Tabla: clientes.csv | Registros: 2000 | Columnas: ['id_cliente', 'ind_empleado', 'pais_residencia', 'sexo', 'fecha_alta', 'conyuemp', 'canal_entrada', 'indfall', 'aparece_en_train', 'aparece_en_test']


,id_cliente,ind_empleado,pais_residencia,sexo,fecha_alta,conyuemp,canal_entrada,indfall,aparece_en_train,aparece_en_test
0,15889,F,ES,V,1995-01-16,N,KAT,N,0,1
1,1050088,N,ES,H,2012-08-10,NaN,KHD,N,1,0


Tabla: provincias.csv | Registros: 52 | Columnas: ['cod_prov', 'nombre_provincia']


,cod_prov,nombre_provincia
0,29,MALAGA
1,13,CIUDAD REAL


Tabla: segmentos.csv | Registros: 3 | Columnas: ['id_segmento', 'descripcion_segmento']


...

,id_cliente,fecha_corte,origen_datos,es_test,age,antiguedad,ind_nuevo,indrel,indrel_1mes,tiprel_1mes,indresi,indext,ind_actividad_cliente,renta,cod_prov,id_segmento
0,1375586,2015-01-28,train,0,35,6,0,1,1.0,A,S,N,1,87218.10,29,1
1,1050611,2015-01-28,train,0,23,35,0,1,1.0,I,S,S,0,35548.74,13,2


Tabla: cliente_producto_mensual.csv | Registros: 16000 | Columnas: ['id_cliente', 'fecha_corte', 'origen_datos', 'id_producto', 'estado_producto']


,id_cliente,fecha_corte,origen_datos,id_producto,estado_producto
0,1375586,2015-01-28,train,1,0
1,1375586,2015-01-28,train,2,0


Tabla: cliente_producto_alta.csv | Registros: 0 | Columnas: ['id_cliente', 'fecha_corte', 'origen_datos', 'id_producto', 'flag_alta_producto']


,id_cliente,fecha_corte,origen_datos,id_producto,flag_alta_producto
